In [2]:
import pandas as pd
df = pd.read_csv('acquiredDataset.csv')

In [3]:

df = df.drop(columns=['attention', 'meditation'])



In [4]:
X = df.drop(columns=['classification'])
y = df['classification']

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_sc = scaler.fit_transform(x_train)
x_test_sc = scaler.transform(x_test)





In [11]:
X_train_cnn = x_train_sc.reshape(-1, 8, 1)
X_test_cnn = x_test_sc.reshape(-1, 8, 1)

In [12]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

In [13]:
cnn_model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(8, 1)),
    BatchNormalization(),
    Dropout(0.3),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    BatchNormalization(),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 6, 64)          │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 6, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 5, 32)          │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 5, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 160)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,137 (59.13 KB)

 Trainable params: 14,945 (58.38 KB)

 Non-trainable params: 192 (768.00 B)

In [15]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_cnn = cnn_model.fit(X_train_cnn, y_train,
                            epochs=100,
                            batch_size=32,
                            validation_split=0.2,
                            callbacks=[early_stop],
                            verbose=1)

Epoch 1/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6130 - loss: 0.6900 - val_accuracy: 0.5485 - val_loss: 0.6752
Epoch 2/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6657 - loss: 0.6148 - val_accuracy: 0.5485 - val_loss: 0.6714
Epoch 3/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6753 - loss: 0.5900 - val_accuracy: 0.5502 - val_loss: 0.6607
Epoch 4/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6925 - loss: 0.5832 - val_accuracy: 0.5702 - val_loss: 0.6463
Epoch 5/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6946 - loss: 0.5714 - val_accuracy: 0.6104 - val_loss: 0.6255
Epoch 6/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6975 - loss: 0.5676 - val_accuracy: 0.6472 - val_loss: 0.5769
Epoch 7/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7088 - loss: 0.5520 - val_accuracy: 0.6304 - val_loss: 0.5889
Epoch 8/100
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7017 - loss: 0.5482 - val_accuracy: 0.6990 

In [17]:
from sklearn.metrics import accuracy_score, classification_report

In [19]:
y_pred_cnn = (cnn_model.predict(X_test_cnn) > 0.5).astype(int)
print("CNN Test Accuracy ", accuracy_score(y_test, y_pred_cnn))

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
CNN Test Accuracy  0.7389558232931727


In [20]:
print(classification_report(y_test, y_pred_cnn))

              precision    recall  f1-score   support

           0       0.78      0.76      0.77       427
           1       0.69      0.72      0.70       320

    accuracy                           0.74       747
   macro avg       0.73      0.74      0.73       747
weighted avg       0.74      0.74      0.74       747

